**main.py — Punto de entrada del EDA sobre el dataset de perfiles profesionales en Data (Stack Overflow 2025)**

Ejecuta el pipeline completo en orden:

1. Carga y limpieza de datos          (notebook_eda_inicial)
2. Análisis univariante               (analisis_univariante)
3. Análisis bivariante                (analisis_bivariante)
4. Análisis multivariante             (analisis_multivariante)

Uso:

    python main.py
    python main.py --step carga
    python main.py --step univariante
    python main.py --step bivariante
    python main.py --step multivariante
    python main.py --no-plots

In [ ]:
import sys
import time
from pathlib import Path


ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path(r"C:\Users\Juanma\GitHub\EDA-1\EDA_Mercado_Laboral_Data")

DATA_DIR    = ROOT / "src" / "data"
UTILS_DIR   = ROOT / "src" / "utils"
RAW_CSV     = DATA_DIR / "develop_dataset_2025.csv"
CLEAN_CSV   = DATA_DIR / "develop_dataset_2025_limpio.csv"
DICT_CSV    = DATA_DIR / "data_dictionary.csv"

if str(UTILS_DIR) not in sys.path:
    sys.path.append(str(UTILS_DIR))

print(f"ROOT: {ROOT}")
print(f"Dataset: {RAW_CSV}")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

from funciones import calidad_datos, shorten_education, score_ed_level


In [ ]:
def _header(title: str) -> None:
    """Imprime una cabecera visual para cada sección."""
    print(f"\n{'=' * 70}")
    print(f"  {title}")
    print(f"{'=' * 70}\n")


def _subheader(title: str) -> None:
    print(f"\n-- {title} {'-' * (65 - len(title))}\n")


def _show(fig: plt.Figure, show_plots: bool) -> None:
    """Muestra o cierra un gráfico según el flag --no-plots."""
    if show_plots:
        plt.show()
    else:
        plt.close(fig)


In [ ]:
def paso_carga(show_plots: bool = True) -> pd.DataFrame:
    _header("PASO 1 - CARGA Y LIMPIEZA DE DATOS")

    _subheader("Carga del dataset original")
    df = pd.read_csv(RAW_CSV, encoding="utf-8")
    print(f"Filas: {df.shape[0]:,} | Columnas: {df.shape[1]}")


    _subheader("Selección de columnas de interés")
    columnas_interes = [
        "ConvertedCompYearly",
        "Age", "EdLevel", "WorkExp", "YearsCode",
        "LanguageHaveWorkedWith",
        "Employment", "AISelect", "RemoteWork",
        "Industry", "Country", "JobSat", "DevType",
    ]
    df_eda = df[columnas_interes].copy()


    _subheader("Filtro por roles del sector Data")
    roles_data = [
        "Data engineer", "Data scientist", "Data or business analyst",
        "Applied scientist", "Database administrator or engineer",
        "AI/ML engineer", "Developer, AI apps or physical AI",
    ]
    patron = "|".join(roles_data)
    df_data = df_eda[df_eda["DevType"].str.contains(patron, case=False, na=False)].copy()
    print(f"Filas totales:     {len(df_eda):,}")
    print(f"Filas roles data:  {len(df_data):,}")
    print(f"Porcentaje:        {len(df_data) / len(df_eda) * 100:.2f}%")


    _subheader("Calidad del subconjunto")
    print(calidad_datos(df_data).to_string())


    _subheader("Limpieza")

    df_data.drop_duplicates(inplace=True)

    cols_constantes = [c for c in df_data.columns if df_data[c].nunique() == 1]
    df_data.drop(columns=cols_constantes, inplace=True)

  
    df_data.dropna(subset=["ConvertedCompYearly"], inplace=True)
    df_data = df_data.dropna(subset=["EdLevel"])
    df_data.drop(columns=["RemoteWork", "LanguageHaveWorkedWith"], inplace=True)


    mapeo_ai = {
        "Yes, I use AI tools daily":                     "Yes",
        "Yes, I use AI tools weekly":                    "Yes",
        "Yes, I use AI tools monthly or infrequently":   "Yes",
        "No, and I don't plan to":                       "No",
        "No, but I plan to soon":                        "No",
    }
    df_data["AISelect"] = df_data["AISelect"].replace(mapeo_ai).fillna("NA")

    mapeo_edlevel = {
        "Bachelor's degree (B.A., B.S., B.Eng., etc.)":                   "Bachelor",
        "Professional degree (JD, MD, Ph.D, Ed.D, etc.)":                 "Professional",
        "Master's degree (M.A., M.S., M.Eng., MBA, etc.)":                "Master",
        "Some college/university study without earning a degree":          "Some College",
        "Associate degree (A.A., A.S., etc.)":                            "Associate",
        "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "Secondary",
        "Primary/elementary school":                                       "Primary",
        "Other (please specify:)":                                         "Other",
    }
    df_data["EdLevel"] = df_data["EdLevel"].replace(mapeo_edlevel)

    mapeo_devtype = {
        "Data or business analyst":              "Data analyst",
        "Database administrator or engineer":    "Database administrator",
        "Developer, AI apps or physical AI":     "AI developer",
    }
    df_data["DevType"] = df_data["DevType"].replace(mapeo_devtype)

    mapeo_country = {
        "United Kingdom of Great Britain and Northern Ireland": "UK and Ireland",
        "United States of America":                             "USA",
        "Iran, Islamic Republic of...":                         "Iran",
        "Hong Kong (S.A.R.)":                                   "Hong Kong",
        "United Arab Emirates":                                 "UAE",
        "Russian Federation":                                   "Russia",
        "Côte d'Ivoire":                                        "Ivory Coast",
        "Congo, Republic of the...":                            "Congo",
    }
    df_data["Country"] = df_data["Country"].replace(mapeo_country)


    df_data["fue_missing_YearCode"] = df_data["YearsCode"].isna()
    df_data["fue_missing_WorkExp"]  = df_data["WorkExp"].isna()
    df_data["fue_missing_Industry"] = df_data["Industry"].isna()

    df_data["Industry"]  = df_data["Industry"].fillna(df_data["Industry"].mode()[0])
    df_data["WorkExp"]   = df_data["WorkExp"].fillna(df_data["WorkExp"].median())
    df_data["YearsCode"] = df_data["YearsCode"].fillna(df_data["YearsCode"].median())


    df_data.loc[df_data["YearsCode"] == 100, "YearsCode"] = pd.NA
    df_data = df_data.dropna(subset=["YearsCode"])

    print(f"Shape tras limpieza: {df_data.shape}")
    print("\nNulos restantes:")
    print(df_data.isnull().sum().sort_values(ascending=False).to_string())


    _subheader("Data dictionary")
    descripciones = {
        "ConvertedCompYearly": "Salario anual convertido a USD",
        "Country":             "País de residencia del encuestado",
        "EdLevel":             "Nivel educativo más alto alcanzado",
        "DevType":             "Tipo de rol o puesto de trabajo",
        "AISelect":            "Uso de herramientas de IA en el trabajo",
        "JobSat":              "Nivel de satisfacción laboral",
        "Industry":            "Sector o industria de la empresa",
        "WorkExp":             "Años de experiencia laboral total",
        "YearsCode":           "Años totales programando (incluye antes de trabajar)",
        "Age":                 "Rango de edad del encuestado",
        "Employment":          "Situación laboral",
    }
    data_dict = pd.DataFrame({
        "columna":         df_data.columns,
        "descripcion_es":  [descripciones.get(c, "Sin descripción") for c in df_data.columns],
        "dtype":           df_data.dtypes.values,
        "nulos_pct":       (df_data.isnull().mean() * 100).round(1).values,
        "unicos":          df_data.nunique().values,
        "ejemplo":         [df_data[c].dropna().iloc[0] if len(df_data[c].dropna()) > 0 else None
                            for c in df_data.columns],
    })
    print(data_dict.to_string(index=False))

    DATA_DIR.mkdir(parents=True, exist_ok=True)
    data_dict.to_csv(DICT_CSV, index=False)
    df_data.to_csv(CLEAN_CSV)
    print(f"\nOK Dataset limpio guardado en: {CLEAN_CSV}")

    return df_data


In [ ]:
def paso_univariante(df: pd.DataFrame, show_plots: bool = True) -> None:
    _header("PASO 2 - ANÁLISIS UNIVARIANTE")

    sns.set_theme(style="whitegrid", palette="Set2")

    numeric_cols     = df.select_dtypes(include="number").columns
    categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns


    _subheader("Tipificación de variables")
    print("Numéricas:    ", list(numeric_cols))
    print("Categóricas:  ", list(categorical_cols))


    _subheader("Tendencia central — variables numéricas")
    print(df[numeric_cols].agg(["mean", "median", "min", "max"]).to_string())


    _subheader("Top 10 salarios más altos")
    print(df.sort_values("ConvertedCompYearly", ascending=False)
            [["ConvertedCompYearly", "Country", "DevType", "WorkExp", "YearsCode"]]
            .head(10).to_string(index=False))


    _subheader("Moda — variables categóricas")
    for col in categorical_cols:
        print(f"  {col:<30} -> {df[col].mode()[0]}")


    _subheader("Frecuencias y distribución — variables categóricas")
    for col in categorical_cols:
        frecuencias = df[col].value_counts(dropna=False)
        porcentajes = (df[col].value_counts(normalize=True, dropna=False) * 100).round(2)
        print(f"\n{col}:")
        print(pd.concat([frecuencias, porcentajes], axis=1, keys=["n", "%"]).head(10).to_string())

        fig, ax = plt.subplots(figsize=(10, 4))
        frecuencias.head(15).sort_values().plot(kind="barh", ax=ax)
        ax.set_title(f"Frequency of {col}")
        ax.set_xlabel("Count")
        plt.tight_layout()
        _show(fig, show_plots)


    _subheader("Estadísticos descriptivos — variables numéricas")
    print(df[numeric_cols].describe().to_string())

    for col in numeric_cols:
        fig, ax = plt.subplots(figsize=(8, 3))
        sns.boxplot(x=df[col], ax=ax)
        ax.set_title(f"Boxplot de {col}")
        plt.tight_layout()
        _show(fig, show_plots)


    _subheader("Medidas de dispersión")
    dispersion = df[numeric_cols].agg(["std", "var", "min", "max"])
    dispersion.loc["range"] = df[numeric_cols].max() - df[numeric_cols].min()
    print(dispersion.to_string())


    _subheader("Distribuciones")
    for col in numeric_cols:
        fig, ax = plt.subplots(figsize=(8, 4))
        data = df[col].dropna()

        if col == "ConvertedCompYearly":
            data = data[data <= data.quantile(0.99)]
            sns.histplot(data, kde=True, bins=30, ax=ax)
            ax.set_title(f"Distribution of {col} (sin top 1%)")
        elif col == "JobSat":
            sns.countplot(x=data, ax=ax)
            ax.set_title(f"Distribution of {col}")
        else:
            sns.histplot(data, kde=True, bins=20, ax=ax)
            ax.set_title(f"Distribution of {col}")

        ax.set_xlabel(col)
        ax.set_ylabel("Frequency")
        plt.tight_layout()
        _show(fig, show_plots)


    _subheader("Outliers por IQR")
    for col in numeric_cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lim_inf = Q1 - 1.5 * IQR
        lim_sup = Q3 + 1.5 * IQR
        n_out = ((df[col] < lim_inf) | (df[col] > lim_sup)).sum()
        pct   = round(n_out / len(df) * 100, 2)
        print(f"  {col:<30} outliers: {n_out:>4}  ({pct}%)  "
              f"límites: [{lim_inf:.1f}, {lim_sup:.1f}]")


In [ ]:
def paso_bivariante(df: pd.DataFrame, show_plots: bool = True) -> None:
    _header("PASO 3 - ANÁLISIS BIVARIANTE")

    sns.set_theme(style="whitegrid", palette="Set2")
    salary_col = "ConvertedCompYearly"


    _subheader("H1 - Edad y salario")
    age_order = [
        "18-24 years old", "25-34 years old", "35-44 years old",
        "45-54 years old", "55-64 years old", "65 years or older", "Prefer not to say",
    ]
    df["Age"] = pd.Categorical(df["Age"], categories=age_order, ordered=True)

    print(df.groupby("Age", observed=True)[salary_col]
            .agg(["count", "mean", "median", "min", "max"])
            .round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    sns.boxplot(data=df, x="Age", y=salary_col, order=age_order, ax=axes[0])
    axes[0].set_yscale("log")
    axes[0].set_title("Salary distribution by age group")
    axes[0].tick_params(axis="x", rotation=40)
    sns.barplot(data=df, x="Age", y=salary_col, order=age_order,
                estimator="median", ax=axes[1])
    axes[1].set_title("Median salary by age group")
    axes[1].tick_params(axis="x", rotation=40)
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H2 - Nivel educativo, empleabilidad y salario")

    tabla_ed = pd.crosstab(df["EdLevel"], df["Employment"], normalize="index") * 100
    print("Distribución de empleo por EdLevel (%):")
    print(tabla_ed.round(2).to_string())

    print("\nMediana salarial por EdLevel:")
    print(df.groupby("EdLevel")[salary_col]
            .agg(["count", "mean", "median"])
            .sort_values("median", ascending=False)
            .round(2).to_string())

    fig, ax = plt.subplots(figsize=(10, 6))
    sns.boxplot(data=df, x=salary_col, y="EdLevel", ax=ax)
    ax.set_xscale("log")
    ax.set_title("Wage distribution by educational attainment")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H3 - Experiencia y salario")

    print("Correlaciones con salario:")
    print(df[["WorkExp", "YearsCode", salary_col]].corr().round(3).to_string())

    df["WorkExp_group"] = pd.cut(
        df["WorkExp"], bins=[0, 5, 10, 20, 35, 60],
        labels=["0-5", "6-10", "11-20", "21-35", "36+"]
    )

    print("\nMediana salarial por rango de experiencia:")
    print(df.groupby("WorkExp_group", observed=True)[salary_col]
            .agg(["count", "mean", "median"])
            .round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    sns.scatterplot(data=df, x="WorkExp", y=salary_col, alpha=0.4, ax=axes[0])
    axes[0].set_yscale("log")
    axes[0].set_title("Work experience vs salary")
    sns.barplot(data=df, x="WorkExp_group", y=salary_col,
                estimator="median", ax=axes[1])
    axes[1].set_title("Median salary by experience group")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H4 - Empleado vs Freelancer")

    df_emp = df[df["Employment"].isin(["Employed", "Freelancer"])].copy()
    print(df_emp.groupby("Employment")[salary_col]
            .agg(["count", "mean", "median", "min", "max"])
            .round(2).to_string())

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.boxplot(data=df_emp, x="Employment", y=salary_col, ax=ax)
    ax.set_yscale("log")
    ax.set_title("Wage distribution: Employed vs Freelancer")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H5 - Uso de IA, salario y satisfacción laboral")

    print(df.groupby("AISelect")[salary_col]
            .agg(["count", "mean", "median"]).round(2).to_string())
    print()
    print(df.groupby("AISelect")["JobSat"]
            .agg(["count", "mean", "median"]).round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    sns.boxplot(data=df, x="AISelect", y=salary_col, ax=axes[0])
    axes[0].set_yscale("log")
    axes[0].set_title("Salary by AISelect")
    sns.boxplot(data=df, x="AISelect", y="JobSat", ax=axes[1])
    axes[1].set_title("Job satisfaction by AISelect")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H6 - Industria y salario")

    salario_ind = (df.groupby("Industry")[salary_col]
                     .agg(["count", "mean", "median"])
                     .sort_values("median", ascending=False)
                     .round(2))
    top_ind = salario_ind[salario_ind["count"] >= 30]
    print(top_ind.to_string())

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    sns.barplot(data=top_ind.reset_index(), x="median", y="Industry", ax=axes[0])
    axes[0].set_title("Ranking of industries by median salary")
    sns.boxplot(data=df, x=salary_col, y="Industry", ax=axes[1])
    axes[1].set_xscale("log")
    axes[1].set_title("Wage distribution by industry")
    plt.tight_layout()
    _show(fig, show_plots)


In [ ]:
def paso_multivariante(df: pd.DataFrame, show_plots: bool = True) -> None:
    _header("PASO 4 - ANÁLISIS MULTIVARIANTE")

    sns.set_theme(style="whitegrid", palette="Set2")
    salary_col     = "ConvertedCompYearly"
    education_col  = "EdLevel"
    employment_col = "Employment"
    workexp_col    = "WorkExp"
    yearscode_col  = "YearsCode"

    df_multi = df.copy()


    exp_bins   = [0, 5, 10, 15, 20, 30, 60]
    exp_labels = ["0-5", "6-10", "11-15", "16-20", "21-30", "31+"]

    df_multi["education_score"] = df_multi[education_col].apply(score_ed_level)
    df_multi["is_employed"]     = df_multi[employment_col].eq("Employed").astype(int)
    df_multi["log_salary"]      = np.log10(df_multi[salary_col])
    df_multi["WorkExp_group"]   = pd.cut(
        df_multi[workexp_col], bins=exp_bins, labels=exp_labels, include_lowest=True
    )


    _subheader("H2.2 - Cualificación académica, empleabilidad y salario")

    empleados_por_ed = (
        df_multi.groupby(education_col)["is_employed"]
        .agg(["count", "mean"])
        .assign(pct_empleados=lambda x: (x["mean"] * 100).round(2))
        .sort_values("pct_empleados", ascending=False)
    )
    print("Empleabilidad por nivel educativo:")
    print(empleados_por_ed.to_string())

    print("\nMediana salarial por nivel educativo:")
    print(df_multi.groupby(education_col)[salary_col]
            .agg(["count", "mean", "median"])
            .sort_values("median", ascending=False)
            .round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.boxplot(data=df_multi, x=salary_col, y=education_col, ax=axes[0])
    axes[0].set_xscale("log")
    axes[0].set_title("Wage distribution by educational attainment")
    sns.barplot(data=empleados_por_ed.reset_index(),
                x="pct_empleados", y=education_col, ax=axes[1])
    axes[1].set_xlim(0, 100)
    axes[1].set_title("Employment rate by educational level")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("H3.2 - Experiencia (laboral + técnica) y salario")

    print("Mediana salarial por rangos de experiencia:")
    print(df_multi.groupby("WorkExp_group", observed=True)[salary_col]
            .agg(["count", "mean", "median"])
            .round(2).to_string())

    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    sns.barplot(data=df_multi, x="WorkExp_group", y=salary_col,
                estimator="median", ax=axes[0])
    axes[0].set_title("Median salary by years of experience")

    sns.scatterplot(data=df_multi, x=workexp_col, y=yearscode_col,
                    size=salary_col, hue="education_score",
                    sizes=(20, 300), alpha=0.5, palette="viridis", ax=axes[1])
    axes[1].set_title("Work exp. vs technical exp. (size = salary)")
    axes[1].legend(title="Ed. score", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("Cruce educación × experiencia (salario y empleabilidad)")

    salary_pivot = df_multi.pivot_table(
        values=salary_col, index=education_col,
        columns="WorkExp_group", aggfunc="median", observed=True
    ).round(2)
    print("Mediana salarial [EdLevel × WorkExp_group]:")
    print(salary_pivot.to_string())

    salary_long = (
        salary_pivot.reset_index()
        .melt(id_vars=education_col, var_name="WorkExp_group", value_name="median_salary")
        .dropna()
    )
    salary_long["Education_short"] = salary_long[education_col].apply(shorten_education)

    fig, ax = plt.subplots(figsize=(13, 6))
    sns.barplot(data=salary_long, x="WorkExp_group", y="median_salary",
                hue="Education_short", ax=ax)
    ax.set_title("Median salary by experience and educational level")
    ax.legend(title="Ed. level", bbox_to_anchor=(1.02, 1), loc="upper left")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("Matriz de correlaciones multivariante")

    corr_vars   = ["log_salary", workexp_col, yearscode_col, "education_score", "is_employed"]
    corr_matrix = df_multi[corr_vars].corr(numeric_only=True).round(3)
    print(corr_matrix.to_string())

    fig, ax = plt.subplots(figsize=(7, 5))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", center=0,
                vmin=-1, vmax=1, ax=ax)
    ax.set_title("Multivariate correlation matrix")
    plt.tight_layout()
    _show(fig, show_plots)


    _subheader("Modelo lineal — salario logarítmico")

    model_data = df_multi[["log_salary", workexp_col, yearscode_col,
                            "education_score", "is_employed"]].dropna()
    y = model_data["log_salary"].values
    X = model_data[[workexp_col, yearscode_col, "education_score", "is_employed"]]
    X_std    = (X - X.mean()) / X.std(ddof=0)
    X_matrix = np.column_stack([np.ones(len(X_std)), X_std.values])
    coef     = np.linalg.lstsq(X_matrix, y, rcond=None)[0]
    pred     = X_matrix @ coef
    r2       = 1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum()

    coef_df = pd.DataFrame({
        "variable":    ["intercept"] + list(X.columns),
        "coeficiente": coef,
    }).round(4)
    print(f"R² del modelo: {r2:.3f}  (n={len(model_data)})")
    print(coef_df.to_string(index=False))


    _subheader("Modelo lineal — empleabilidad")

    emp_data = df_multi[["is_employed", workexp_col, yearscode_col, "education_score"]].dropna()
    y_e      = emp_data["is_employed"].values
    X_e      = emp_data[[workexp_col, yearscode_col, "education_score"]]
    X_e_std  = (X_e - X_e.mean()) / X_e.std(ddof=0)
    X_e_mat  = np.column_stack([np.ones(len(X_e_std)), X_e_std.values])
    coef_e   = np.linalg.lstsq(X_e_mat, y_e, rcond=None)[0]
    pred_e   = X_e_mat @ coef_e
    r2_e     = 1 - ((y_e - pred_e) ** 2).sum() / ((y_e - y_e.mean()) ** 2).sum()

    coef_e_df = pd.DataFrame({
        "variable":    ["intercept"] + list(X_e.columns),
        "coeficiente": coef_e,
    }).round(4)
    print(f"R² del modelo: {r2_e:.3f}  (n={len(emp_data)})")
    print(coef_e_df.to_string(index=False))


In [ ]:
PASOS = {
    "carga":          paso_carga,
    "univariante":    paso_univariante,
    "bivariante":     paso_bivariante,
    "multivariante":  paso_multivariante,
}


def run_pipeline(step: str | None = None, show_plots: bool = True) -> None:
    """Orquesta el EDA completo o un paso concreto."""
    t0 = time.time()

    if step and step not in PASOS:
        print(f"[ERROR] Paso desconocido: '{step}'. Opciones: {list(PASOS)}")
        sys.exit(1)


    if step and step != "carga":
        if not CLEAN_CSV.exists():
            print(f"[INFO] {CLEAN_CSV} no encontrado. Ejecutando paso de carga primero.")
            df = paso_carga(show_plots)
        else:
            df = pd.read_csv(CLEAN_CSV, index_col=0)
        PASOS[step](df, show_plots)

    elif step == "carga":
        paso_carga(show_plots)

    else:
        # Pipeline completo
        df = paso_carga(show_plots)
        paso_univariante(df, show_plots)
        paso_bivariante(df, show_plots)
        paso_multivariante(df, show_plots)

    elapsed = time.time() - t0
    print(f"\n{'=' * 70}")
    print(f"  OK Pipeline completado en {elapsed:.1f}s")
    print(f"{'=' * 70}\n")


In [ ]:
# Celda de ejecucion manual para VS Code / Jupyter.
# Cambia RUN_PIPELINE a True solo cuando quieras lanzar el analisis.
# STEP puede ser: None, "carga", "univariante", "bivariante" o "multivariante".

RUN_PIPELINE = True
STEP = "carga"
SHOW_PLOTS = False

if RUN_PIPELINE:
    run_pipeline(step=STEP, show_plots=SHOW_PLOTS)
else:
    print("Pipeline no ejecutado. Pon RUN_PIPELINE = True para lanzarlo.")
